# Task 1 — MAE self-supervised adaptation

Branch `task1`. The challenge is few-shot by design: 10 labelled samples, and no
more coming. But `H_test.mat` ships **20 unlabelled** CSI samples, and labels are
the scarce resource here, not CSI.

This runs WiFo2's own masked-autoencoder objective over all 30 samples with no
labels, so the encoder adapts to this deployment before the classifier is fitted.
The repo has every piece — `forward_encoder`, `forward_decoder`, `decoder_pred`,
`forward_loss` — but never wires them into a training loop; `main.py` only runs
the supervised task heads.

Scored under the challenge metric: **binary F1, LoS positive**.

In [ ]:
import sys, warnings, json
from pathlib import Path

warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, torch, scipy.io as sio, h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

from main import freeze_wifo
from physics_features import extract
import mae_pretrain as mae

SEEDS = [0, 1, 7, 13, 42, 99, 123, 2024, 31337, 555]
LOS = 1                      # class 1 = LoS, established in notebook/task1_physics.ipynb
W  = ROOT / 'weights/model_best.pkl'
FD = ROOT / 'weights/FastDepthV2_L1_Best.pth'

def metrics(yt, yp):
    return {'f1': f1_score(yt, yp, pos_label=LOS, zero_division=0),
            'precision': precision_score(yt, yp, pos_label=LOS, zero_division=0),
            'recall': recall_score(yt, yp, pos_label=LOS, zero_division=0),
            'macro_f1': f1_score(yt, yp, average='macro')}

## 1. The unlabelled corpus

30 samples: the 10 labelled training ones plus the 20 unlabelled test ones. Only
the CSI is used — `L_train.mat` is never opened in this section.

Using the test *inputs* is transductive self-supervised learning: legitimate,
since the test CSI is given, and no test labels exist to leak.

In [ ]:
def load_mat(path, key):
    try:
        return sio.loadmat(path)[key]
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            a = f[key][...]
        if a.dtype.names == ('real', 'imag'):
            a = a['real'] + 1j * a['imag']
        return a.transpose(range(a.ndim)[::-1])

def to_x(Hc):
    return torch.stack([torch.tensor(Hc.real), torch.tensor(Hc.imag)], 1).float()

H    = load_mat(ROOT / 'dataset/Task1/H_train.mat', 'H')
y    = load_mat(ROOT / 'dataset/Task1/L_train.mat', 'label').ravel().astype(int)
H_te = load_mat(ROOT / 'dataset/Task1/H_test.mat', 'H')

x_tr, x_te = to_x(H), to_x(H_te)
corpus = torch.cat([x_tr, x_te])

print(f'labelled   {tuple(x_tr.shape)}  labels {y}')
print(f'unlabelled {tuple(x_te.shape)}')
print(f'MAE corpus {tuple(corpus.shape)}  <- 3x the labelled data, no labels used')

## 2. What one MAE step does

`fre` masking at ratio 0.5 hides half the 384 tokens along the frequency axis.
The encoder sees only the survivors; the decoder gets mask tokens in the gaps and
must reconstruct the hidden patches. Loss is MSE on real and imag, **computed on
the masked patches only** (`forward_loss` returns masked and visible separately).

In [ ]:
m, args = mae.build(W, FD)
masked, visible = mae.mae_loss(m, corpus[:8])
print(f'mask_ratio {mae.MASK_RATIO}, strategy {mae.MASK_STRATEGY!r}')
print(f'  loss on masked patches  {masked.item():.5f}   <- the training signal')
print(f'  loss on visible patches {visible.item():.5f}   <- reported, not optimised')

lat, mask, _, input_size, _ = m.forward_encoder(corpus[:8], mae.MASK_RATIO, mae.MASK_STRATEGY, scale=[1,1,1])
print(f'\npatch grid {input_size} = 384 tokens; encoder sees {lat.shape[1]} of them')
print(f'trainable during MAE: encoder blocks, decoder blocks, embeddings, norms, mask token')

## 3. How long to adapt

The tradeoff: too little and nothing changes, too much and 30 samples overwrite
the large-scale pretraining. Measured, physics + WiFo2 at k=3:

| MAE epochs | recon loss | binary F1 |
|---|---|---|
| 0 (no MAE) | — | 0.846 |
| 5 | 0.804 | 0.875 |
| **10 (default)** | 0.778 | **0.875** |
| 20 | 0.739 | 0.914 |
| 40 | 0.696 | 0.814 |
| 80 | 0.659 | 0.814 |

Identical downstream score across 5 different MAE init seeds at each row — but
**the batch ordering matters**. The table above seeded before constructing the
model; seeding after (what `mae_pretrain.pretrain` does) shuffles the corpus
differently and gives 0.904 at 10 epochs instead of 0.875. Both beat the 0.846
baseline, but a gain whose size moves 0.03 on incidental RNG is not a precise
measurement.

**`DEFAULT_EPOCHS = 10`, not the 20 that peaks.** 20 sits on a spike whose
neighbours are 0.04-0.10 lower, and it was chosen by reading this same metric —
the selection leakage that cost 0.08 in `task1_physics.ipynb`. 10 is on the flat
part and its gain (+0.029) is the honest one. Past 40 epochs the adaptation
overfits and drops below the no-MAE baseline.

In [ ]:
m, args = mae.build(W, FD)
hist = mae.pretrain(m, corpus, epochs=mae.DEFAULT_EPOCHS, log=2)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(hist)+1), hist, marker='o')
ax.set(xlabel='epoch', ylabel='masked reconstruction loss',
       title=f'MAE on {len(corpus)} unlabelled CSI samples')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. Does it help the classifier?

Same pipeline as `task1_physics.ipynb`: physics features concatenated with
mean-pooled WiFo2 features, `SelectKBest` inside the fold, logistic regression.
Only the WiFo2 half changes.

In [ ]:
X_phys, _    = extract(H)
X_phys_te, _ = extract(H_te)

base, _ = mae.build(W, FD)
freeze_wifo(args, base, task_id=1); base.eval()
V_base,    V_base_te    = mae.features(base, x_tr), mae.features(base, x_te)
V_adapted, V_adapted_te = mae.features(m, x_tr),    mae.features(m, x_te)

def evaluate(X, k=3):
    pipe = lambda: make_pipeline(StandardScaler(), SelectKBest(f_classif, k=min(k, X.shape[1])),
                                 LogisticRegression(C=1, max_iter=3000))
    scores = []
    for s in SEEDS:
        oof = np.zeros(len(y), int)
        for itr, iva in StratifiedKFold(5, shuffle=True, random_state=s).split(X, y):
            p = pipe(); p.fit(X[itr], y[itr]); oof[iva] = p.predict(X[iva])
        scores.append(metrics(y, oof)['f1'])
    oof = np.zeros(len(y), int)
    for itr, iva in LeaveOneOut().split(X):
        p = pipe(); p.fit(X[itr], y[itr]); oof[iva] = p.predict(X[iva])
    return np.mean(scores), np.std(scores), metrics(y, oof), oof

sets = {
    'wifo alone, no MAE'      : V_base,
    'wifo alone, MAE'         : V_adapted,
    'physics + wifo, no MAE'  : np.hstack([X_phys, V_base]),
    'physics + wifo, MAE'     : np.hstack([X_phys, V_adapted]),
}
print(f"{'configuration':26s}{'binF1':>8s}{'std':>7s}{'|':>3s}{'LOO F1':>8s}{'prec':>7s}{'rec':>7s}")
out = {}
for name, X in sets.items():
    mean, std, lo, oof = evaluate(X)
    out[name] = (mean, std, lo, oof)
    print(f"{name:26s}{mean:>8.3f}{std:>7.3f}{'|':>3s}{lo['f1']:>8.3f}"
          f"{lo['precision']:>7.3f}{lo['recall']:>7.3f}")

d = out['physics + wifo, MAE'][0] - out['physics + wifo, no MAE'][0]
print(f'\nMAE effect on the best configuration: {d:+.3f}')
print(f"CV noise (std) is {out['physics + wifo, MAE'][1]:.3f} - treat a gain below that as unproven.")

## 5. Predictions and submission

In [ ]:
X_all    = np.hstack([X_phys,    V_adapted])
X_all_te = np.hstack([X_phys_te, V_adapted_te])

final = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=3),
                      LogisticRegression(C=1, max_iter=3000))
final.fit(X_all, y)
pred_te = final.predict(X_all_te).astype(int)
prob_te = final.predict_proba(X_all_te)[:, list(final.classes_).index(LOS)]

print(f"{'sample':>7}{'pred':>7}{'P(LoS)':>9}")
for i in range(len(pred_te)):
    print(f'{i:>7}{pred_te[i]:>7}{prob_te[i]:>9.3f}')
print(f'\npredicted balance {np.bincount(pred_te, minlength=2)}  (train {np.bincount(y)})')

out_dir = ROOT / 'experiments/task1_mae'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'submission.json').write_text(json.dumps(
    {'task1': pred_te.tolist(), 'task2': [], 'task3': []}))

sd = {k: v for k, v in m.state_dict().items() if not k.startswith(('res18.', 'fast_depth.'))}
torch.save(sd, out_dir / 'model_best.pkl')

mean, std, lo, _ = out['physics + wifo, MAE']
(out_dir / 'cv_results.json').write_text(json.dumps({
    'metric': 'binary F1, LoS positive (challenge spec)',
    'mae_epochs': mae.DEFAULT_EPOCHS, 'mae_recon_loss': float(hist[-1]),
    'cv5_mean': float(mean), 'cv5_std': float(std),
    'loocv': {a: float(b) for a, b in lo.items()},
    'variants': {k: float(v[0]) for k, v in out.items()},
}, indent=2))

print(f"\ntask1 -> {pred_te.tolist()}")
print('Run make_submission.py to attach the task2 persistence baseline.')

## Summary

| configuration | binary F1 | std |
|---|---|---|
| WiFo2 alone, no MAE | 0.679 | 0.122 |
| WiFo2 alone, MAE 10ep | 0.714 | 0.122 |
| physics + WiFo2, no MAE | 0.846 | 0.032 |
| **physics + WiFo2, MAE 10ep** | **0.904** | 0.085 |

MAE is the first idea on this branch that did not come back negative —
augmentation, threshold tuning and the Doppler/angular features all did.

What is solid:

- reconstruction loss falls 0.812 -> 0.770 on 30 unlabelled samples, so the
  objective is learning
- the gain is positive in every configuration tried, and MAE also lifts WiFo2
  alone (0.679 -> 0.714), so it is not an artefact of the physics half
- over-adaptation is real and measurable: past 40 epochs the score drops below
  the no-MAE baseline

What is not solid:

- **+0.057 against a CV std of 0.085.** Below the noise
- the exact value moves with MAE batch ordering (0.875 to 0.904 at the same 10
  epochs, depending only on when the RNG is seeded)
- CV std nearly triples versus no-MAE (0.032 -> 0.085): the adapted features are
  less stable across fold splits
- LOOCV is **unchanged** at 0.857 with the same confusion — the 5-fold gain comes
  from fold splits LOOCV does not exercise
- the local CV has already been shown to predict the private ranking poorly: a
  +0.68 local gain previously bought +0.03 on the leaderboard

Worth a submission slot only if one is spare. The standing 0.75 submission uses
the no-MAE model; predictions here differ from it on 3 of 20 samples (indices 8, 15, 17).